# 04 · 香港天文台每日天气｜成员 D
样本期为 2023-02-06 至 2026-10-31。原始数据保存在 `data/raw/weather/`。
已有冻结文件时，直接从本地读取。需要重新取数时，在新副本中运行 Notebook。

In [ ]:
import sys
from pathlib import Path
from time import sleep

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.fetch_utils import get_text
from src.weather_features import RAW_WEATHER, WEATHER_CSV, build_weather_features, check_weather

RAW_WEATHER.mkdir(parents=True, exist_ok=True)


def download_if_missing(filename, url, params=None):
    path = RAW_WEATHER / filename
    if path.exists():
        return path
    text = get_text(url, params=params)
    if text is None:
        raise RuntimeError(f"Download failed: {filename}")
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    temporary.replace(path)
    sleep(1)
    return path

## 气温和雨量
路线①。最高气温通过 CLMMAXT API 分年请求。雨量从 DATA.GOV.HK 公布的 HKO 站全年 CSV 下载。
冻结的 CSV 只到 2026 年 9 月。10 月使用天文台同站逐日观测表，以 HTML 表格中的最高气温和总雨量补齐。

In [ ]:
climate = "https://data.weather.gov.hk/weatherAPI"
for year in range(2023, 2027):
    download_if_missing(
        f"hko_tmax_{year}.csv",
        f"{climate}/opendata/opendata.php",
        {"dataType": "CLMMAXT", "year": year, "rformat": "csv", "station": "HKO"},
    )
download_if_missing("hko_rainfall.csv", f"{climate}/cis/csvfile/HKO/ALL/daily_HKO_RF_ALL.csv")
download_if_missing("metob202610.htm", "https://www.weather.gov.hk/wxinfo/pastwx/metob202610.htm");

## 8 号或以上风球、红雨与黑雨
路线②。警告数据库页面的 HTML 只有查询表单，日期记录由 JavaScript 请求 `tc.dat` 和 `rstorm.dat` 后生成。
一次下载每个档案，再按香港日期检查每个警告的生效区间。跨午夜仍生效的警告会标记两天。

In [ ]:
warning_db = "https://www.weather.gov.hk/dps/wxinfo/climat/warndb"
for filename in ("tc.dat", "rstorm.dat"):
    download_if_missing(filename, f"{warning_db}/{filename}")

## 生成天气日表
三层检查包括手算时间格式、全样本日期与数值范围、官方记录的现场对照。
第三层核对 2023-09-01 的苏拉风球和 2023-09-08 的黑雨，详见 `docs/member_d.md`。

In [ ]:
weather = build_weather_features()
check_weather(weather)
WEATHER_CSV.parent.mkdir(parents=True, exist_ok=True)
weather.to_csv(WEATHER_CSV, index=False)
print(f"{WEATHER_CSV.name}: {len(weather):,} days")
weather.set_index("Date").loc[["2023-09-01", "2023-09-08", "2026-10-31"]]